In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import compose_windows, make_names_dict

from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc, accuracy_score

## Time-specific classifiers:

### This section trains time-specific (3 possible timepoints, 3 tissues) classifiers for chromatin loop formation prediction

1. Trains cross-validated classifiers and plots their ROC curves (with mean and std)
2. For each time window: generates a combined plot with ROC curves of the best CV classifier for each tissue
3. For each configuration of (timepoint, tissue) it saves two models:
    - best cross-validation classifier
    - classifier fitted to the entire data

In [ ]:
def time_specific_with_cv_and_roc_plots(classifier, windows=["06-08", "10-12", "14-16"], tissues=["Neuroblasts", "Neurons", "Glia"], n_splits=10):

    model_names_dict = make_names_dict()
    model_name = model_names_dict[type(classifier).__name__]['full']
    short = model_names_dict[type(classifier).__name__]['short']

    for w in windows:
        
        X = pd.read_csv(f"data/training/hrs{w}/data_diff_hrs{w}.csv", index_col=0)
        
        # Prepare the per-window ROC figure
        #plt.figure(figsize=(8, 8))
        tissue_plotting_info = {t: {} for t in tissues}

        for t in tissues:
            y_path = f"data/training/hrs{w}/y_{t}.csv"
            if not os.path.exists(y_path):
                continue
            y = pd.read_csv(y_path, index_col=0).iloc[:, 0]

            # Align X and y just in case
            X = X.loc[y.index]

            # Cross-validation
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)
            probs, trues, accs = [], [], []
            fold_index = {}

            for (i, (train_idx, test_idx)) in enumerate(kf.split(X)):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
                fold_index[i] = (train_idx, test_idx)

                classifier = clone(classifier) # this creates an UNFITTED model with the same parameters as before
                classifier.fit(X_train, y_train)

                p = classifier.predict_proba(X_test)[:, 1]
                probs.append(p)
                trues.append(y_test.values)
                accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

            # Compute ROC curve and other metrics
            mean_acc = np.mean(accs)
            std_acc = np.std(accs)

            fprs, tprs, roc_aucs = [], [], []
            for (true, prob) in zip(trues, probs):
                fpr, tpr, _ = roc_curve(true, prob)
                fprs.append(fpr)
                tprs.append(tpr)
                roc_aucs.append(auc(fpr, tpr))

            # Use linear interpolation to plot mean +/- 1 std. bounds
            mean_fpr = np.linspace(0, 1, 100)
            interp_tprs = []

            _, ax = plt.subplots(figsize=(6, 6))

            for idx in range(n_splits):
                interp_tpr = np.interp(mean_fpr, fprs[idx], tprs[idx])
                interp_tpr[0] = 0.0
                interp_tprs.append(interp_tpr)

            mean_tpr = np.mean(interp_tprs, axis=0)
            mean_tpr[-1] = 1.0
            mean_auc = auc(mean_fpr, mean_tpr)
            std_auc = np.std(roc_aucs)

            ax.plot(
                mean_fpr,
                mean_tpr,
                label=f"Mean ROC (AUC = {mean_auc:.3f} "+ r'$\pm$' + f" {std_auc:.3f})",
                lw=1,
                alpha=0.8
            )
            plt.grid(axis='both')
            std_tpr = np.std(interp_tprs, axis=0)
            tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
            tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
            ax.fill_between(
                mean_fpr,
                tprs_lower,
                tprs_upper,
                alpha=0.2,
                label=r"$\pm$ 1 std. dev.",
            )
            ax.set(
                xlabel="False Positive Rate",
                ylabel="True Positive Rate",
                title=f"{model_name} ROC - {t}, hrs{w}\nAUC = {mean_auc:.3f} "+ r'$\pm$' + f" {std_auc:.3f}\nAcc = {mean_acc:.3f} " + r'$\pm$' + f" {std_acc:.3f}",
            )

            plt.plot([0, 1], [0, 1], "k--", lw=1)
            ax.legend(loc="lower right")

            # save the figure
            parent_dir = f"results/figures/time_specific/{short}"
            os.makedirs(parent_dir, exist_ok=True)
            out_path = f"{parent_dir}/roc_{short}_{t}_hrs{w}.png"
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close()

            # SAVE THE BEST MODELS
            ## Find and train best (according to ROC AUC) classifier from cross-validation
            best_fold = np.argmax(roc_aucs)
            (train_idx, test_idx) = fold_index[best_fold]
            X_train = X.iloc[train_idx]
            y_train = y.iloc[train_idx]

            cv_best_classifier = clone(classifier)
            cv_best_classifier.fit(X_train, y_train)

            cv_dir = f"results/models/time_specific/cv"
            cv_path = f"{cv_dir}/{short}_{t}_{w}.pkl"
            os.makedirs(cv_dir, exist_ok=True)
            pickle.dump(cv_best_classifier, open(cv_path, "wb"))

            ## Train a separate model, on all of the data
            all_data_classifier = clone(classifier)
            all_data_classifier.fit(X, y)

            all_data_dir = f"results/models/time_specific/all_data"
            all_data_path = f"{all_data_dir}/{short}_{t}_{w}.pkl"
            os.makedirs(all_data_dir, exist_ok=True)
            pickle.dump(all_data_classifier, open(all_data_path, "wb"))

            # Record info for per-window plot
            for key in ['mean_fpr', 'mean_tpr', 'std_tpr', 'tprs_upper', 'tprs_lower', 'mean_auc', 'mean_acc', 'std_auc', 'std_acc']:
                exec(f"tissue_plotting_info['{t}']['{key}'] = {key}")

            print(f"Trained {model_name} for {t} at hrs{w}: ROC curve saved at {out_path}")
        
        plt.figure(figsize=(8, 8))
        # Generate per-window combined plot
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title=f"Mean {model_name} ROC curves, hrs{w}",
        )
        for t in tissues:
            ax.plot(    
                tissue_plotting_info[t]['mean_fpr'],
                tissue_plotting_info[t]['mean_tpr'],
                label=f"Mean ROC for {t} (AUC = {tissue_plotting_info[t]['mean_auc']:.3f} "+ r'$\pm$'+ f" {tissue_plotting_info[t]['std_auc']:.3f})",
                lw=1,
                alpha=0.8
            )

            plt.grid(axis='both')
            
            ax.fill_between(
                tissue_plotting_info[t]['mean_fpr'],
                tissue_plotting_info[t]['tprs_lower'],
                tissue_plotting_info[t]['tprs_upper'],
                alpha=0.2,
            )


        plt.plot([0, 1], [0, 1], "k--", lw=1)
        ax.legend(loc="lower right")   

        # save the figure
        parent_dir = f"results/figures/time_specific/{short}"
        os.makedirs(parent_dir, exist_ok=True)
        out_path = f"{parent_dir}/roc_{short}_combined_hrs{w}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()

        print(f"Saved {model_name} ROC plot for window {w}: {out_path}")

### Random Forest classifiers

In [ ]:
classifier = RandomForestClassifier(
                n_estimators=500,
                random_state=0,
                n_jobs=-1
            )

time_specific_with_cv_and_roc_plots(classifier=classifier)

### Support Vector Machine

In [ ]:
classifier = SVC(probability=True)

time_specific_with_cv_and_roc_plots(classifier=classifier)

### Logistic Regression (unregularized)

In [ ]:
classifier = LogisticRegression(C=np.inf)

time_specific_with_cv_and_roc_plots(classifier=classifier)

### XGBoost

In [ ]:
classifier = XGBClassifier(
    n_estimators = 500,
    n_jobs = -1
)

time_specific_with_cv_and_roc_plots(classifier=classifier)

## Time-agnostic classifiers

### This section trains time-agnostic (3 tissues) classifiers for chromatin loop formation prediction

1. Trains cross-validated classifiers and plots their ROC curves (with mean and std)
2. Generates a combined plot with ROC curves of the best CV classifier for each tissue
3. For each tissue it saves two models:
    - best cross-validation classifier
    - classifier fitted to the entire data
4. For each tissue: compares the performance of different architectures

### Brudnopis
Funkcja(`klasyfikator`):
1. Dla tkanki: ładuje dataframe ze wszystkich trzech okien
2. łączenie i etykietowanie krotkami (okno, klasa)
3. Potem chyba wszystko tak samo, tylko że ze `StratifiedKFold`
 - czyli wykres z CV dla każdej tkanki (3 osobne)
 - jeden wspólny wykres dla tkanek (1 wykres, trzy krzywe) <- *słownik!!!*
 - mogę zrobić `json.dump` tego słownika dziwnego i wczytać go w drugiej analizie *WOW*
4. Będę nadal chciał mieć ten tkankowy słownik do plotowania, tylko teraz wykres combined będzie **dla architektury** a trzy krzywe będą pochodzić **od tkanek**

Potem, co nawet ważniejsze, porównujemy architektury wewnątrz tkanki. To będą 3 wykresy po 4 krzywe.
 - chcemy mieć mean i std z CV, więc nie możemy po prostu wczytywać modelu z .pkl
 - wczytać `json` jako słownik plotowania (kluczami są tkanki, głębszymi kluczami są poszczególne zmienne) --> wtedy mam jeden `json` dla jednej architektury

```{python}
for t in tissues:
    wczytuje słownik modelowy[t]
    robię wykres dla tkanki
```


dodać combined pomiędzy tkankami na końcu tej funkcji!

In [ ]:
from utils import compose_windows, make_names_dict

def time_agnostic_with_cv_and_roc_plots(classifier, windows=["06-08", "10-12", "14-16"], tissues=["Neuroblasts", "Neurons", "Glia"], n_splits=10):
    """
    Train tissue-specific, time-agnostic, cross-validated classifiers and save them. Generate ROC curve for each tissue with CV mean and std.
    args:
        classifier: one of a few predefined sklearn classifier objects
    returns:
        tissue_plotting_info: dictionary with information for downstream analyses
    """
    model_names_dict = make_names_dict()
    model_name = model_names_dict[type(classifier).__name__]['full']
    short = model_names_dict[type(classifier).__name__]['short']
    
    tissue_plotting_info = {t: {} for t in tissues}

    for t in tissues:
        # Load windows
        (X, y, composite) = compose_windows(t)

        # Cross-validation
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
        probs, trues, accs = [], [], []
        fold_index = {}

        for (i, (train_idx, test_idx)) in enumerate(skf.split(X, composite)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            fold_index[i] = (train_idx, test_idx)

            classifier = clone(classifier) # this creates an UNFITTED model with the same parameters as before
            classifier.fit(X_train, y_train)

            p = classifier.predict_proba(X_test)[:, 1]
            probs.append(p)
            trues.append(y_test.values)
            accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

        # Compute ROC curve and other metrics
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)

        fprs, tprs, roc_aucs = [], [], []
        for (true, prob) in zip(trues, probs):
            fpr, tpr, _ = roc_curve(true, prob)
            fprs.append(fpr)
            tprs.append(tpr)
            roc_aucs.append(auc(fpr, tpr))

        # Use linear interpolation to plot mean +/- 1 std. bounds
        mean_fpr = np.linspace(0, 1, 100)
        interp_tprs = []

        _, ax = plt.subplots(figsize=(6, 6))

        for idx in range(n_splits):
            interp_tpr = np.interp(mean_fpr, fprs[idx], tprs[idx])
            interp_tpr[0] = 0.0
            interp_tprs.append(interp_tpr)

        mean_tpr = np.mean(interp_tprs, axis=0)
        mean_tpr[-1] = 1.0
        mean_auc = auc(mean_fpr, mean_tpr)
        std_auc = np.std(roc_aucs)

        ax.plot(
            mean_fpr,
            mean_tpr,
            label=f"Mean ROC (AUC = {mean_auc:.3f} "+ r'$\pm$' + f" {std_auc:.3f})",
            lw=1,
            alpha=0.8
        )
        plt.grid(axis='both')
        std_tpr = np.std(interp_tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        ax.fill_between(
            mean_fpr,
            tprs_lower,
            tprs_upper,
            alpha=0.2,
            label=r"$\pm$ 1 std. dev.",
        )
        ax.set(
            xlabel="False Positive Rate",
            ylabel="True Positive Rate",
            title=f"Time-agnostic {model_name} ROC ({t})\nAUC = {mean_auc:.3f} "+ r'$\pm$' + f" {std_auc:.3f}\nAcc = {mean_acc:.3f} " + r'$\pm$' + f" {std_acc:.3f}",
        )

        plt.plot([0, 1], [0, 1], "k--", lw=1)
        ax.legend(loc="lower right")

        # save the figure
        parent_dir = f"results/figures/time_agnostic/{short}"
        os.makedirs(parent_dir, exist_ok=True)
        out_path = f"{parent_dir}/roc_{short}_{t}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()

        # SAVE THE BEST MODELS
        ## Find and train best (according to ROC AUC) classifier from cross-validation
        best_fold = np.argmax(roc_aucs)
        (train_idx, test_idx) = fold_index[best_fold]
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        cv_best_classifier = clone(classifier)
        cv_best_classifier.fit(X_train, y_train)

        cv_dir = f"results/models/time_agnostic/cv"
        cv_path = f"{cv_dir}/{short}_{t}.pkl"
        os.makedirs(cv_dir, exist_ok=True)
        pickle.dump(cv_best_classifier, open(cv_path, "wb"))

        ## Train a separate model, on all of the data
        all_data_classifier = clone(classifier)
        all_data_classifier.fit(X, y)

        all_data_dir = f"results/models/time_agnostic/all_data"
        all_data_path = f"{all_data_dir}/{short}_{t}.pkl"
        os.makedirs(all_data_dir, exist_ok=True)
        pickle.dump(all_data_classifier, open(all_data_path, "wb"))

        # IMPORTANT!
        # Record info for per-window plot
        for key in ['mean_fpr', 'mean_tpr', 'tprs_upper', 'tprs_lower', 'mean_auc', 'mean_acc', 'std_auc', 'std_acc']:
            exec(f"tissue_plotting_info['{t}']['{key}'] = {key}")

        print(f"Trained {model_name} for {t}: ROC curve saved at {out_path}")

    # FIXME: share this info in a more sensible way (like json dump)
    return tissue_plotting_info


### Time-agnostic **Random Forest**

In [ ]:
classifier = RandomForestClassifier(
                n_estimators=500,
                random_state=0,
                n_jobs=-1
            )

rf_dict = time_agnostic_with_cv_and_roc_plots(classifier=classifier)

### Time-agnostic **Support Vector Machine**

In [ ]:
classifier = SVC(probability=True)

svm_dict = time_agnostic_with_cv_and_roc_plots(classifier=classifier)

### Time-agnostic **Logistic Regression** (unregularized)

In [ ]:
classifier = LogisticRegression(C=np.inf)

lr_dict = time_agnostic_with_cv_and_roc_plots(classifier=classifier)

### Time-agnostic **XGBoost**

In [ ]:
classifier = XGBClassifier(
    n_estimators = 500,
    n_jobs = -1
)

xgb_dict = time_agnostic_with_cv_and_roc_plots(classifier=classifier)

## Grouped bar plot of mean AUC $\pm$ 1 std.

### Grouped by model

In [ ]:
# this cell requires prior running of the Time-agnostic section
tissues = ["Neuroblasts", "Neurons", "Glia"]
model_names = ["RF", "SVM", "LR", "XGB"]
model_dicts = [rf_dict, svm_dict, lr_dict, xgb_dict]

values = []
for t in tissues:
    values += [[model_name, t, model_dict[t]["mean_auc"], model_dict[t]["std_auc"]] for (model_name, model_dict) in zip(model_names, model_dicts)]
    
df = pd.DataFrame(values,
                    index=model_names*len(tissues),
                    columns=["model_name", "tissue", "mean_auc", "std_auc"])

x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
for t_idx, t in enumerate(tissues):

    means, stds = [], []
    for m_idx, model in enumerate(model_names):
        row = df[(df['model_name'] == model) & (df['tissue'] == t)]
        means.append(row['mean_auc'].values[0]) if not row.empty else 0
        stds.append(row['std_auc'].values[0]) if not row.empty else 0
        ax.errorbar(x[m_idx] + (t_idx - 1) * width, means[m_idx], yerr=stds[m_idx], fmt='.' ,capsize=6, color="black", alpha=0.8)
    ax.bar(x + (t_idx - 1) * width, means, width, label=str(t), alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel(f'ROC AUC')
ax.set_xlabel('Model architecture')
ax.set_title(r'A bar plot of mean cross-validation ROC AUC $\pm$ 1 standard deviation,'+'\ngrouped by model architecture')
ax.legend(title='Tissue')
plt.tight_layout()
plt.ylim(0,1)

# save the figure
parent_dir = "results/figures/AUC_bar_plot"
os.makedirs(parent_dir, exist_ok=True)
out_path = f"{parent_dir}/per_model.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

### Grouped by tissue

In [ ]:
tissues = ["Neuroblasts", "Neurons", "Glia"]
model_names = ["RF", "SVM", "LR", "XGB"]
model_dicts = [rf_dict, svm_dict, lr_dict, xgb_dict]

values = []
for t in tissues:
    values += [[model_name, t, model_dict[t]["mean_auc"], model_dict[t]["std_auc"]] for (model_name, model_dict) in zip(model_names, model_dicts)]
    
df = pd.DataFrame(values,
                    index=model_names*len(tissues),
                    columns=["model_name", "tissue", "mean_auc", "std_auc"])

x = np.arange(len(tissues))
width = 0.15

colors = {
    "RF": "#130dabff",   # light blue
    "SVM": "#ff0303",   # medium blue
    "LR": "#ff8c00",   # dark blue
    "XGB": "#45c9ed",   # dark blue
}

fig, ax = plt.subplots(figsize=(12, 6))

for m_idx, model in enumerate(model_names):

    means, stds = [], []
    for t_idx, t in enumerate(tissues):
        row = df[(df['model_name'] == model) & (df['tissue'] == t)]
        means.append(row['mean_auc'].values[0]) if not row.empty else 0
        stds.append(row['std_auc'].values[0]) if not row.empty else 0
        ax.errorbar(x[t_idx] + (m_idx - 1) * width, means[t_idx], yerr=stds[t_idx], fmt='.' ,capsize=6, color="black", alpha=0.8)
    ax.bar(x + (m_idx - 1) * width, means, width, label=str(model), alpha=0.8, color=colors[model])
    
ax.set_xticks(x+width/2)
ax.set_xticklabels(tissues)
ax.set_ylabel(f'ROC AUC')
ax.set_xlabel('Tissue')
ax.set_title(r'A bar plot of mean cross-validation ROC AUC $\pm$ 1 standard deviation,'+'\ngrouped by tissue')
ax.legend(title='Model architecture')
plt.tight_layout()
plt.ylim(0,1)

# save the figure
parent_dir = "results/figures/AUC_bar_plot"
os.makedirs(parent_dir, exist_ok=True)
out_path = f"{parent_dir}/per_tissue.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()